# Knowledge Context Pack v2 — Graph Reasoning Design

**Date:** 2026-06-18

**Status:** draft design spec

**Scope:** `spur-mcp`, `spur-analyst`, `spur-cli analyst build`, `crates/spur-context/poc/duckdb-analyst/*.sql`

## Objective

Design the next version of `knowledge_context_pack`: a semantic-answer evidence pack that uses semantic retrieval as the front door, exact graph tools as source-of-truth grounding, DuckPGQ for explainable paths, and Onager for whole-graph ranking/community signals.

The goal is not to make DuckPGQ or Onager answer natural-language questions directly. The goal is to make semantic answers more grounded, explainable, and risk-aware by enriching retrieved candidates with graph paths, centrality, community, churn, and confidence evidence.

## Current State

SPUR already has the required substrate:

| Layer | Current capability |
|---|---|
| Graph artifact | `NodeKind`, `RelationKind`, `GraphEdgeKind`, stable symbol IDs, resolved and unresolved edges, temporal facts. |
| Analyst DuckDB | `_meta`, `nodes`, `edges`, `edges_unresolved`, temporal views, scorecard views, FTS search macros. |
| DuckPGQ | Property-graph path queries over materialized compatibility tables. |
| Onager | PageRank, components, Louvain communities, density, degree materializations. |
| Exact graph MCP | `code_symbol_search`, `code_read_symbol`, `code_callers`, `code_callees`, `code_subgraph`. |
| Current `knowledge_context_pack` | BM25/hybrid candidates, scorecard signals, small exact graph context, staleness metadata, next-tool selectors. |

Live local analyst DB sampled during design:

- graph hash: `cf835ad0c8f2546b9f39a686c7970748053a68a34287a062682aeade9d08c7f4`
- schema: `spur-graph-schema-v9`
- nodes: `53,389`
- resolved edges: `106,190`
- unresolved edges: `111,762`
- commits: `3,393`
- symbol snapshots: `784,427`
- temporal edges: `813,968`

The missing piece is not data. It is a query-time orchestration layer that turns this data into a bounded semantic answer pack.

## Core Claim

Knowledge Context Pack v2 should answer this class of question:

> “Given a natural-language engineering question, what are the most relevant local facts, how are they connected, how important/risky are they, and what exact evidence should the agent read next?”

The answer is a structured evidence pack, not a free-form final answer. An LLM or agent can then synthesize prose from the pack.

### Pipeline

```text
natural-language question
→ semantic retrieval finds candidate docs/symbols
→ exact graph resolves current source identities
→ DuckPGQ explains paths between candidates and anchors
→ Onager ranks importance, communities, and choke points
→ temporal/co-change views add change-risk context
→ pack builder returns bounded evidence with caveats and next tools
```

This keeps semantic matching approximate and graph evidence explicit.

## Target Use Cases

| Use case | Example question | KCP v2 answer shape |
|---|---|---|
| Semantic architecture explanation | “How does delegation approval work?” | Primary docs/symbols, shortest workflow paths, central handlers, unresolved caveats. |
| Risk-aware change review | “Why is changing this handler risky?” | Direct impact, PageRank/posture, hot callers, path to side-effect anchors. |
| Reading path | “What should I read to understand task dispatch?” | Ordered docs/symbols with dependency paths and centrality weighting. |
| Boundary analysis | “Does TUI code reach project-management mutation?” | DuckPGQ path proof or no-path result, edge confidence, boundary labels. |
| Task split advice | “This feature touches these files; should it be split?” | Communities/components touched, co-change clusters, split candidates. |
| Resolver quality triage | “Where is the graph weakest around notebook lineage?” | Unresolved hotspots, heuristic edges, diagnostics, affected communities. |
| Notebook/data lineage explanation | “What feeds this port or datasource?” | Cell/port paths, produces/consumes/binds/emits facts, source cells/docs. |

The first two should drive the MVP because they map directly to SPUR’s worker/reviewer loop.

## Design Goals And Non-Goals

### Goals

1. Return answer-ready evidence for semantic questions without requiring manual tool chaining.
2. Preserve exact stable IDs, file paths, graph hash, and staleness metadata.
3. Explain why a result matters using path, centrality, community, churn, and confidence signals.
4. Keep packs bounded by budget and intent.
5. Make graph algorithms useful through named concepts, not generic SQL output.
6. Degrade safely when analyst DB, DuckPGQ, Onager, Lance, or temporal data are unavailable.

### Non-Goals

- No compiler-grade type resolution.
- No local-variable or dataflow semantics beyond existing graph facts.
- No free-form answer generation inside the MCP tool.
- No requirement that all semantic questions be answerable.
- No replacement for exact `code_*` follow-up tools.
- No broad SCIP parity or new ontology variants in this design.

## Architecture

```mermaid
flowchart TB
  User[Agent question] --> API[knowledge_context_pack_v2]

  API --> Retrieval[Semantic candidate retrieval]
  Retrieval --> BM25[Analyst BM25/search macros]
  Retrieval --> Hybrid[Optional Lance hybrid search]

  API --> Ground[Exact grounding]
  Ground --> Exact[code_* graph handlers]

  API --> Paths[Path reasoning]
  Paths --> DuckPGQ[DuckPGQ shortest paths / reachability]

  API --> Rank[Whole-graph signals]
  Rank --> Onager[Onager PageRank/components/communities]

  API --> Temporal[Temporal and co-change signals]
  Temporal --> Analyst[Analyst DuckDB views]

  BM25 --> Pack[Evidence pack builder]
  Hybrid --> Pack
  Exact --> Pack
  DuckPGQ --> Pack
  Onager --> Pack
  Analyst --> Pack

  Pack --> Agent[Agent synthesizes answer]

  classDef api fill:#fef9c3,stroke:#ca8a04,color:#111827;
  classDef engine fill:#ecfdf5,stroke:#059669,color:#111827;
  classDef data fill:#eff6ff,stroke:#2563eb,color:#111827;
  class API,Pack api;
  class Retrieval,Ground,Paths,Rank,Temporal engine;
  class BM25,Hybrid,Exact,DuckPGQ,Onager,Analyst data;
```

`knowledge_context_pack_v2` should live in `spur-mcp` because that is where worker agents already access exact graph tools. The analyst SQL remains the warm analytical substrate.

## Algorithmic Enrichers

### DuckPGQ Enrichers

DuckPGQ should be used when a semantic question needs an explainable route:

| Enricher | Inputs | Output |
|---|---|---|
| `shortest_paths_between_candidates` | top symbols/docs converted to graph anchors | 1-3 hop paths with edge kinds and confidence. |
| `paths_to_risk_anchors` | candidate symbols + configured anchors | paths to mutation, event emission, worktree, cost, license, or PM surfaces. |
| `boundary_violation_paths` | source/target layer rules | concrete forbidden/discouraged paths. |
| `impact_path_sample` | changed symbol + impacted symbols | representative paths explaining why an item is in scope. |

### Onager Enrichers

Onager should be used when the question needs ranking or graph structure:

| Enricher | Inputs | Output |
|---|---|---|
| `centrality_summary` | candidate stable IDs | PageRank, in/out degree, posture. |
| `community_summary` | candidate stable IDs | community IDs, component sizes, cross-community spread. |
| `choke_point_score` | candidate stable IDs | bridge/bottleneck approximation from degree/component/path metrics. |
| `task_split_signal` | changed files/symbols | number of communities/components touched and suggested grouping. |
| `dead_island_signal` | graph components | isolated low-churn components likely stale or orphaned. |

KCP v2 should expose these as named evidence sections, not raw graph-algorithm jargon.

## Proposed Request Contract

```json
{
  "query": "How does delegation approval work and what makes it risky to change?",
  "intent": "explain|change|review|debug|plan",
  "scope": "all|docs|code|graph",
  "limit": 8,
  "include_tests": true,
  "max_symbol_bodies": 3,
  "graph_reasoning": {
    "paths": true,
    "communities": true,
    "risk": true,
    "max_path_hops": 4,
    "max_paths": 6
  },
  "anchors": ["worktree_mutation", "beads_mutation", "event_emission"]
}
```

Defaults should be conservative:

- `paths = true` only for `change`, `review`, and `debug`; optional for `explain`.
- `communities = true` only when there are at least two grounded code candidates.
- `risk = true` for all code-containing scopes.
- `max_path_hops = 4`.
- Popular sinks are boundaries, not expansion roots.

## Proposed Response Shape

```json
{
  "query": "How does delegation approval work?",
  "intent": "explain",
  "answerable": true,
  "confidence": "high",
  "graph_content_hash": "cf835ad0...",
  "staleness": {
    "analyst_matches_exact_graph": true,
    "response_file_oids_match": true
  },
  "summary_hint": "Approval flows through plan/task state, review decision handling, and mutation/audit emission.",
  "primary_evidence": [],
  "supporting_docs": [],
  "graph_paths": [],
  "risk_scorecard": [],
  "community_context": [],
  "temporal_context": [],
  "caveats": [],
  "recommended_next_tools": []
}
```

### New Evidence Sections

| Section | Purpose |
|---|---|
| `graph_paths` | Explain connections between top candidates and risk anchors. |
| `risk_scorecard` | Centrality, churn, caller count, unresolved density, posture. |
| `community_context` | Components/communities touched; whether a question crosses subsystem boundaries. |
| `temporal_context` | Recent churn, fix hotspots, co-change files. |
| `caveats` | Staleness, heuristic edges, unresolved labels, popular-sink truncation, unavailable engines. |

The response must remain evidence-first. It should not claim behavioral correctness.

## Semantic Answer Policy

KCP v2 should support semantic answers by providing structured evidence. The agent synthesizing the answer must follow these rules:

1. Treat BM25/hybrid hits as candidates, not proof.
2. Treat exact graph symbol reads and current file OIDs as grounding.
3. Prefer shortest path explanations over broad subgraph dumps.
4. Report unresolved or heuristic edges when they affect the answer.
5. Distinguish “no path found” from “path search unavailable or budget-limited.”
6. Use Onager ranks as prioritization signals, not correctness claims.
7. Include recommended exact follow-ups for any high-impact conclusion.

### Answerable Classes

KCP v2 should answer well:

- “Where does this concept live?”
- “What code and docs explain this workflow?”
- “Why is this change risky?”
- “What subsystem/community owns this behavior?”
- “What exact path connects these two concepts?”

KCP v2 should mark low confidence for:

- runtime behavior requiring execution traces
- type inference or local variable dataflow
- correctness claims without tests/specs
- concepts absent from indexed docs/symbol text

## SQL Surface Requirements

The design assumes existing or future analyst views/macros can provide these named surfaces:

| Surface | Required? | Notes |
|---|---:|---|
| `search_context_candidates` | yes | Existing candidate source for docs/code. |
| `search_context_candidates_hybrid` | optional | Existing hybrid route when Lance/query vector is available. |
| `search_graph` | optional | Existing graph-expanded search macro; useful but not enough for explainable paths. |
| `v_symbol_scorecard` | yes | Current centrality/churn/posture base. |
| `v_symbol_inbound` | yes | Caller/import/container counts. |
| `v_unresolved_hotspots` | yes | Resolver caveats. |
| `v_graph_metrics` | yes | Whole-graph health signals. |
| `v_community_modules` | new | Community/component rollup with dominant crates/files. |
| `v_choke_points` | new | High bridge/bottleneck candidates. |
| `v_task_split_candidates` | new | Group changed files by community/co-change/static edges. |
| `path_shortest_between(a, b, max_hops)` | new | DuckPGQ-backed path macro/table function. |
| `paths_to_anchor(seed, anchor_set, max_hops)` | new | DuckPGQ-backed risk-anchor path macro. |

The MVP can use only existing surfaces plus a small path macro. Deeper algorithmic surfaces can follow.

## Example Semantic Pack

Question:

> “What should I read to understand task dispatch, and where are the risky edges?”

Expected KCP v2 structure:

1. `primary_evidence`
   - task dispatch handlers/services
   - relevant plan/reconciler docs
   - MCP tool symbols
2. `graph_paths`
   - submit plan path to dispatch/audit/comment creation
   - dispatch path to worker completion/review handling
3. `risk_scorecard`
   - central symbols with caller count, PageRank, churn, posture
4. `community_context`
   - communities touched by dispatch, review, PM mutation, ACP events
5. `temporal_context`
   - recently changed files and fix hotspots in the flow
6. `caveats`
   - unresolved dynamic calls, macro edges, stale analyst DB if applicable
7. `recommended_next_tools`
   - exact `code_read_symbol` selectors for the top handlers
   - `code_callers` selectors for high-risk public entry points

A final LLM answer can then produce a reading order and a risk summary with evidence.

## Failure Modes And Mitigations

| Failure mode | Mitigation |
|---|---|
| Semantic retrieval returns on-topic docs but poor code hits | Keep docs and code evidence separate; allow doc-only high confidence for conceptual questions, require exact graph follow-up for code claims. |
| DuckPGQ path query expands through popular sinks | Use precomputed sink boundaries and max-hop/max-path caps; never expand through known response builders or pervasive utilities by default. |
| Onager rank makes irrelevant symbols look important | Display rank as signal, not proof; require lexical/semantic relevance before graph ranking can boost a candidate. |
| Analyst DB is stale | Expose staleness and downgrade source bodies; re-ground with exact `code_*` tools. |
| Path not found because graph is incomplete | Return `no_path_with_caveat`, including unresolved-label samples and graph scope limits. |
| Communities are unstable across builds | Treat community IDs as ephemeral labels; describe dominant files/crates instead of relying on numeric IDs. |
| Pack grows too large | Hard caps per section, path count, and neighbor rows; include next-tool selectors for deeper exploration. |

## Rollout Plan

### Phase 1 — Path-Aware Pack

- Add optional `graph_paths` to `knowledge_context_pack`.
- Use candidate stable IDs from existing retrieval.
- Add one DuckPGQ shortest-path macro or Rust-side query wrapper.
- Return path rows with edge kind, relation, confidence, bind method, and hop count.

### Phase 2 — Algorithm-Aware Pack

- Add `community_context` and stronger `risk_scorecard` from existing Onager materializations.
- Add community rollup view if needed.
- Add task-split signal for changed file sets.

### Phase 3 — Semantic Answer Evaluation

- Build fixture questions with expected evidence, not expected prose.
- Measure whether KCP v2 retrieves the right docs/symbols/paths under a fixed budget.
- Track false positives from path expansion and graph ranking.

### Phase 4 — Agent Adoption

- Update code-explore guidance to prefer KCP v2 for semantic questions.
- Teach agents to use path/risk/community sections as evidence, not as final answers.
- Keep exact `code_*` tools as required follow-up for high-impact claims.

## Acceptance Criteria

KCP v2 is ready when:

- A natural-language question returns docs, symbols, paths, and risk signals in one bounded response.
- Every code evidence item includes stable symbol ID, file path, graph hash, and grounding method.
- DuckPGQ path results include relation, edge kind, confidence, bind method, and path budget metadata.
- Onager-derived fields are clearly labeled as prioritization signals.
- Missing DuckPGQ/Onager/Lance/temporal data degrades with explicit caveats instead of failing the whole pack.
- Fixture tests assert evidence structure for at least five semantic question classes.
- Existing `knowledge_context_pack` callers can opt into graph reasoning without breaking the current response contract.

## Recommendation

Treat this as `knowledge_context_pack` evolution, not a new standalone product.

The MVP should add path-aware evidence first because it directly improves semantic answer explainability. Onager community/risk enrichment should follow immediately after, because it turns retrieved candidates into prioritized reading and review guidance.